3a (Reddit-only variant, no NVD needed): correlate each BAT construct's monthly rate against the others, and against a binary "month contains a major named event" indicator — does EX lag COG by a month, for instance, directly answering your "open question" about whether exhaustion lags differently than cognitive impairment.


Detrending method: I'll use a rolling-mean subtraction (12-month centered moving average) rather than a linear detrend, since your trend isn't obviously linear — it's flatter 2018-2020, then rises faster 2021-2023, then plateaus 2024-2026. A 12-month rolling mean removes seasonal/secular drift without assuming a specific functional form. The result is a "residual" series — how far each month deviates from its local trend — which is what gets cross-correlated.
Lag-sign convention (important to state explicitly so results aren't misread): I'll define a positive lag for series B relative to A as "B lags A" — meaning A's value at month t is compared to B's value at month t+lag. So if EX leads COG by 2 months, you'd see the strongest correlation at lag = +2 when correlating EX (as the leading series) against COG.
Event indicator: a simple binary monthly flag = 1 if a named event's disclosure date falls in that month, 0 otherwise, built from the

In [1]:
!pip3 install matplotlib

Defaulting to user installation because normal site-packages is not writeable


In [2]:
"""
Analysis 3a: Cross-Correlation (Reddit-only, no NVD download needed)

PURPOSE
-------
Using the monthly_trend.csv produced by Analysis 1, this script:

  1. Detrends each construct's monthly rate series (12-month centered
     rolling mean subtraction) to remove the secular ~38% rise found in
     Analysis 1, so we're correlating short-term fluctuations, not both
     series independently drifting upward over years.

  2. Computes pairwise cross-correlation functions (CCF) between all
     6 construct pairs (EX-EMO, EX-COG, EX-MD, EMO-COG, EMO-MD, COG-MD)
     across lags of -6 to +6 months. This directly answers the project's
     open question: "does exhaustion lag differently than cognitive
     impairment?"

  3. Builds a binary monthly "major event occurred" indicator from the
     4 confirmed event dates (Log4Shell, SolarWinds, CrowdStrike, MOVEit)
     and cross-correlates each detrended construct series against it,
     to see whether named events precede/follow construct-specific spikes.

LAG-SIGN CONVENTION (read this before interpreting output)
------------------------------------------------------------
For a pair (A, B), a POSITIVE lag means: A's value at month t is compared
to B's value at month (t + lag). In plain language: at lag = +2, we're
asking "does A's level 2 months ago predict B's level now?" -- i.e.
A LEADS B by 2 months, or equivalently B LAGS A by 2 months.
So the strongest correlation at a positive lag means the FIRST-named
series in the pair leads; at a negative lag, the SECOND-named series leads.

INPUT
-----
monthly_trend.csv (from analysis1_monthly_trend.py)

OUTPUT
------
- ccf_construct_pairs.csv     : full CCF table for all construct pairs
- ccf_event_indicator.csv     : CCF of each construct against the event flag
- ccf_summary.png             : grid of CCF plots (construct pairs + event)
- Printed console summary: peak lag and peak correlation for each pair,
  plus significance threshold lines (approx 95% CI for white noise:
  +/- 1.96/sqrt(N))
"""

import pandas as pd
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

# ============================================================
# CONFIG
# ============================================================
INPUT_CSV = "monthly_trend.csv"
CONSTRUCT_RATE_COLS = ["EX_yes_rate", "EMO_yes_rate", "COG_yes_rate", "MD_yes_rate"]
ROLLING_WINDOW = 12          # months, for detrending
MAX_LAG = 6                  # months, +/- range for CCF

# Confirmed event dates (from Step 2 research)
EVENTS = {
    "Log4Shell": "2021-12-09",
    "SolarWinds": "2020-12-08",
    "CrowdStrike": "2024-07-19",
    "MOVEit": "2023-05-31",
}

OUTPUT_PAIRS_CSV = "ccf_construct_pairs.csv"
OUTPUT_EVENT_CSV = "ccf_event_indicator.csv"
OUTPUT_PNG = "ccf_summary.png"


def load_and_detrend(path):
    df = pd.read_csv(path)
    df["year_month"] = pd.to_datetime(df["year_month"])
    df = df.sort_values("year_month").reset_index(drop=True)
    print(f"Loaded {len(df)} months: {df['year_month'].min().strftime('%Y-%m')} "
          f"to {df['year_month'].max().strftime('%Y-%m')}")

    for col in CONSTRUCT_RATE_COLS:
        if col not in df.columns:
            print(f"WARNING: '{col}' not found in {path}.")
            continue
        trend = df[col].rolling(window=ROLLING_WINDOW, center=True, min_periods=1).mean()
        df[f"{col}_detrended"] = df[col] - trend

    return df


def build_event_indicator(df):
    event_months = set()
    for name, date_str in EVENTS.items():
        period = pd.Timestamp(date_str).to_period("M")
        event_months.add(period)

    df["event_flag"] = df["year_month"].dt.to_period("M").apply(
        lambda p: 1 if p in event_months else 0
    )
    n_flagged = df["event_flag"].sum()
    print(f"Event indicator: {n_flagged} month(s) flagged out of {len(df)} "
          f"(expected {len(EVENTS)} -- one per event, unless two events "
          f"fall in the same calendar month).")
    return df


def ccf(x, y, max_lag):
    """
    Cross-correlation of x and y across lags -max_lag..+max_lag.
    Positive lag k: corr(x[t], y[t+k]) -- x leads y by k months.
    Both series are standardized (z-scored) before correlating, and NaNs
    (from rolling-window edges) are pairwise-dropped per lag.
    """
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)
    n = len(x)
    results = []
    for lag in range(-max_lag, max_lag + 1):
        if lag >= 0:
            x_seg = x[: n - lag] if lag > 0 else x
            y_seg = y[lag:] if lag > 0 else y
        else:
            x_seg = x[-lag:]
            y_seg = y[: n + lag]

        mask = ~(np.isnan(x_seg) | np.isnan(y_seg))
        n_valid = mask.sum()
        if n_valid < 8:  # too few points to trust a correlation
            results.append({"lag": lag, "corr": np.nan, "n_valid": int(n_valid)})
            continue

        x_clean = x_seg[mask]
        y_clean = y_seg[mask]
        if x_clean.std() == 0 or y_clean.std() == 0:
            results.append({"lag": lag, "corr": np.nan, "n_valid": int(n_valid)})
            continue

        r = np.corrcoef(x_clean, y_clean)[0, 1]
        results.append({"lag": lag, "corr": round(r, 4), "n_valid": int(n_valid)})

    return pd.DataFrame(results)


def run_construct_pairs(df):
    constructs = [c.replace("_yes_rate", "") for c in CONSTRUCT_RATE_COLS if f"{c}_detrended" in df.columns]
    pairs = []
    for i, a in enumerate(constructs):
        for b in constructs[i + 1:]:
            pairs.append((a, b))

    all_rows = []
    print("\n" + "=" * 70)
    print("CONSTRUCT-PAIR CROSS-CORRELATION (detrended series)")
    print("=" * 70)

    n_months = df["year_month"].notna().sum()
    sig_threshold = round(1.96 / np.sqrt(n_months), 4)
    print(f"Approx. 95% significance threshold for |r| (white-noise null, "
          f"N={n_months} months): {sig_threshold}\n")

    for a, b in pairs:
        col_a = f"{a}_yes_rate_detrended"
        col_b = f"{b}_yes_rate_detrended"
        result = ccf(df[col_a], df[col_b], MAX_LAG)
        result["pair"] = f"{a}-{b}"
        all_rows.append(result)

        peak_row = result.loc[result["corr"].abs().idxmax()] if result["corr"].notna().any() else None
        if peak_row is not None:
            direction = (f"{a} leads {b}" if peak_row["lag"] > 0
                         else f"{b} leads {a}" if peak_row["lag"] < 0
                         else "simultaneous (lag 0)")
            sig_flag = "  <-- exceeds white-noise threshold" if abs(peak_row["corr"]) > sig_threshold else ""
            print(f"{a:5s} vs {b:5s} | peak lag = {int(peak_row['lag']):+d} months | "
                  f"peak r = {peak_row['corr']:+.3f} | {direction}{sig_flag}")

    combined = pd.concat(all_rows, ignore_index=True)
    combined.to_csv(OUTPUT_PAIRS_CSV, index=False)
    print(f"\nSaved full CCF table -> {OUTPUT_PAIRS_CSV}")
    return combined, pairs, sig_threshold


def run_event_indicator(df):
    constructs = [c.replace("_yes_rate", "") for c in CONSTRUCT_RATE_COLS if f"{c}_detrended" in df.columns]
    all_rows = []
    print("\n" + "=" * 70)
    print("CONSTRUCT vs EVENT INDICATOR CROSS-CORRELATION (detrended)")
    print("=" * 70)
    print("Positive lag: construct LEADS the event-month flag (rises before).")
    print("Negative lag: construct LAGS the event-month flag (rises after) --")
    print("this is the 'hangover' signature if present.\n")

    for c in constructs:
        col = f"{c}_yes_rate_detrended"
        result = ccf(df[col], df["event_flag"], MAX_LAG)
        result["construct"] = c
        all_rows.append(result)

        peak_row = result.loc[result["corr"].abs().idxmax()] if result["corr"].notna().any() else None
        if peak_row is not None:
            lag = int(peak_row["lag"])
            if lag > 0:
                interp = f"{c} tends to rise {lag} month(s) BEFORE named events (anticipatory / unrelated)"
            elif lag < 0:
                interp = f"{c} tends to rise {abs(lag)} month(s) AFTER named events (consistent with hangover)"
            else:
                interp = f"{c} moves simultaneously with named events"
            print(f"{c:5s} | peak lag = {lag:+d} | peak r = {peak_row['corr']:+.3f} | {interp}")

    combined = pd.concat(all_rows, ignore_index=True)
    combined.to_csv(OUTPUT_EVENT_CSV, index=False)
    print(f"\nSaved event-indicator CCF table -> {OUTPUT_EVENT_CSV}")
    return combined, constructs


def plot_all(pair_df, pairs, event_df, constructs, sig_threshold, out_path):
    n_pair_plots = len(pairs)
    n_event_plots = len(constructs)
    n_total = n_pair_plots + n_event_plots
    n_cols = 4
    n_rows = int(np.ceil(n_total / n_cols))

    fig, axes = plt.subplots(n_rows, n_cols, figsize=(4 * n_cols, 3.2 * n_rows))
    axes = axes.flatten()

    plot_idx = 0
    for a, b in pairs:
        sub = pair_df[pair_df["pair"] == f"{a}-{b}"]
        ax = axes[plot_idx]
        ax.bar(sub["lag"], sub["corr"], color="#2563eb", width=0.7)
        ax.axhline(0, color="black", linewidth=0.8)
        ax.axhline(sig_threshold, color="red", linestyle="--", linewidth=0.7)
        ax.axhline(-sig_threshold, color="red", linestyle="--", linewidth=0.7)
        ax.set_title(f"{a} vs {b}", fontsize=10)
        ax.set_xlabel("lag (months)", fontsize=8)
        ax.tick_params(labelsize=8)
        plot_idx += 1

    for c in constructs:
        sub = event_df[event_df["construct"] == c]
        ax = axes[plot_idx]
        ax.bar(sub["lag"], sub["corr"], color="#dc2626", width=0.7)
        ax.axhline(0, color="black", linewidth=0.8)
        ax.axhline(sig_threshold, color="red", linestyle="--", linewidth=0.7)
        ax.axhline(-sig_threshold, color="red", linestyle="--", linewidth=0.7)
        ax.set_title(f"{c} vs event flag", fontsize=10)
        ax.set_xlabel("lag (months)", fontsize=8)
        ax.tick_params(labelsize=8)
        plot_idx += 1

    for j in range(plot_idx, len(axes)):
        axes[j].axis("off")

    plt.tight_layout()
    plt.savefig(out_path, dpi=150)
    print(f"\nSaved chart -> {out_path}")


if __name__ == "__main__":
    df = load_and_detrend(INPUT_CSV)
    df = build_event_indicator(df)

    pair_df, pairs, sig_threshold = run_construct_pairs(df)
    event_df, constructs = run_event_indicator(df)

    plot_all(pair_df, pairs, event_df, constructs, sig_threshold, OUTPUT_PNG)

    print("\n" + "=" * 70)
    print("INTERPRETATION NOTE")
    print("=" * 70)
    print("With only 4 events in ~100 months, the event-indicator CCF has")
    print("very limited statistical power (the event_flag series is mostly")
    print("zeros). Treat those results as exploratory/illustrative, not")
    print("confirmatory -- they're a complement to the event-anchored window")
    print("analysis (Analysis 2/4), not a replacement for it.")

Loaded 100 months: 2018-01 to 2026-04
Event indicator: 4 month(s) flagged out of 100 (expected 4 -- one per event, unless two events fall in the same calendar month).

CONSTRUCT-PAIR CROSS-CORRELATION (detrended series)
Approx. 95% significance threshold for |r| (white-noise null, N=100 months): 0.196

EX    vs EMO   | peak lag = +0 months | peak r = +0.478 | simultaneous (lag 0)  <-- exceeds white-noise threshold
EX    vs COG   | peak lag = +0 months | peak r = +0.220 | simultaneous (lag 0)  <-- exceeds white-noise threshold
EX    vs MD    | peak lag = +0 months | peak r = +0.384 | simultaneous (lag 0)  <-- exceeds white-noise threshold
EMO   vs COG   | peak lag = +0 months | peak r = +0.242 | simultaneous (lag 0)  <-- exceeds white-noise threshold
EMO   vs MD    | peak lag = +0 months | peak r = +0.469 | simultaneous (lag 0)  <-- exceeds white-noise threshold
COG   vs MD    | peak lag = +5 months | peak r = -0.156 | COG leads MD

Saved full CCF table -> ccf_construct_pairs.csv

CONST